In [240]:
import P04_Parser
from igor2 import binarywave
import numpy as np
import matplotlib.pyplot as plt
import re
import hyperspy.api as hs
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path

def load_ibw_swept_as_kmap(file):
    """Swept (2D) .ibw cut -> Kmap with 3 identical theta_y slices, so P04_Parser plotting works."""
    wave = binarywave.load(file)["wave"]
    data = np.array(wave["wData"], dtype=float)     # (energy, angle)
    sfa = wave["wave_header"]["sfA"]
    sfb = wave["wave_header"]["sfB"]
    energy = sfb[0] + sfa[0] * np.arange(data.shape[0])
    theta = sfb[1] + sfa[1] * np.arange(data.shape[1])
    data_map = np.repeat(data[:, :, np.newaxis], 3, axis=2)
    return P04_Parser.Kmap(data_map, energy, theta, np.array([-1, 0, 1]))


In [241]:
plt.rcParams.update({
    "figure.dpi": 110,        # scales the whole figure (text, lines, markers) together
    "font.size": 16,          # base size, also used by checkbox and slider labels
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
    "lines.linewidth": 2,
})


In [242]:
import os
import re

file_dir = r"T:\Beamtimes\MaxIV\Oct_26\raw\FeSi_2A_day3_temp_series_LV_49" #FeSi_2B_49eV_LV_temp_series" #FeSi_2A_day3_temp_series_LV_49"
files = os.listdir(file_dir)

files_names_fesi = [f for f in files if f.endswith(".ibw") and "testing" not in f and f.startswith("TaFoil")]

# Sort by temperature, with each "down" scan right after its "up" scan
def sort_key(f):
    T = float(re.search(r"_(\d+(?:\.\d+)?)K", f).group(1))
    return (T, "down" in f.lower())

files_names_fesi = sorted(files_names_fesi, key=sort_key)
files = [os.path.join(file_dir, f) for f in files_names_fesi]

def temp_from_filename(f):
    return str(re.search(r"_(\d+(?:\.\d+)?)K", f).group(1)) + "K" + (" down" if "down" in f.lower() else "")

labels = {f: temp_from_filename(f) for f in files_names_fesi} #[20:-14] for FeSi and [19:-17] for TaFoil
labels

{'TaFoil_swept_hv_49_LV_GX_20K_0001Swept.ibw': '20K',
 'TaFoil_swept_hv_49_LV_GX_20K_down_0001Swept.ibw': '20K down',
 'TaFoil_swept_hv_49_LV_GX_40K_0001Swept.ibw': '40K',
 'TaFoil_swept_hv_49_LV_GX_40K_down_0001Swept.ibw': '40K down',
 'TaFoil_swept_hv_49_LV_GX_60K_0001Swept.ibw': '60K'}

In [244]:
#Fermi level fitting
%matplotlib qt
from matplotlib.widgets import CheckButtons

from scipy.ndimage import gaussian_filter1d, zoom
from P04_Parser.util_funcs import (
    fit_2d_polynomial,
    evaluate_2d_polynomial,
    fermi_dirac,
    fermi_lin,
    angle_to_kx_deflector,
    angle_to_ky_deflector,
)
from scipy.optimize import curve_fit

colors = ["black", "navy", "cornflowerblue", "hotpink", "red", "darkorange", "gold"]
temp_to_index = {20: 0, 30: 1, 40: 2, 50: 3, 60: 4, 100: 5, 200: 6}


def line_style(file):
    name = Path(file).name
    T = temp_to_index[int(re.search(r"_(\d+(?:\.\d+)?)K", name).group(1))]
    return dict(
        color=colors[T],
        linestyle="--" if "down" in name.lower() else "-",
    )

fig, ax = plt.subplots(figsize=(14, 8))
fig.subplots_adjust(right=0.85)


fermi_levels = []
curve_lines = {}  # label -> [data line, fit line], toggled together by the check buttons

for f in files:
    fix_temp=sort_key(f)[0]

    data = load_ibw_swept_as_kmap(f)

    edc = data.get_EDC_inside(roi = hs.roi.CircleROI(cx=0, cy=0, r=15))
    span = hs.roi.SpanROI(left=-100, right=150)

    sig = span(edc)

    len_kin_en = sig.axes_manager[0].size
    offset_kin_en = sig.axes_manager[0].offset
    scale_kin_en = sig.axes_manager[0].scale

    kinetic_energies = np.linspace(
        offset_kin_en, offset_kin_en + scale_kin_en * len_kin_en, len_kin_en
    )

    intensity_data = sig.data
    
    intensity_data = (intensity_data - np.min(intensity_data)) / (np.max(intensity_data) - np.min(intensity_data))

    kinetic_energies, intensity_data = kinetic_energies[300:], intensity_data[300:]

    def gauss_fermi_lin_fixed_temp(x, Gnd, Scale, Lin, Ef, sigma):
        kT = fix_temp * 8.6173303e-5

        return gaussian_filter1d(fermi_lin(x, Gnd, Scale, Lin, Ef, kT), sigma=sigma)

    fit_guess = (
                intensity_data[-1],
                intensity_data[0],
                intensity_data[1] - intensity_data[0],
                kinetic_energies[len(kinetic_energies) // 2],
                10,
    )
    fit_func = gauss_fermi_lin_fixed_temp
    bounds = (
        (0, 0, -np.inf, -np.inf, 0),
        (np.inf, np.inf, np.inf, np.inf, np.inf),
    )

    popt, pcov = curve_fit(
        fit_func, kinetic_energies, intensity_data, p0=fit_guess, bounds=bounds, absolute_sigma=False
    )



    # plt.figure()
    (data_line,) = ax.plot(kinetic_energies, intensity_data,
            label=labels[Path(f).name],
            **line_style(f),)
    kin_en_highres = np.linspace(
        kinetic_energies[0], kinetic_energies[-1], 1000
    )

    ## The sigma broadening in gaussian_filter1d works in pixel by pixel, so when we want to plot higher
    ## resolution, eg 1000 datapoints here, we need to scale up the sigma acordingly.
    (fit_line,) = ax.plot(
        kin_en_highres,
        fit_func(
            kin_en_highres, *popt[:-1], popt[-1] * 1000 / len(kinetic_energies)
        ),
        # label=labels[Path(f).name] + "fit",
        alpha=0.5,
        **line_style(f),
    )
    curve_lines[labels[Path(f).name]] = [data_line, fit_line]
    fermi_levels.append([popt[3], np.sqrt(pcov[3, 3]), labels[Path(f).name]])
    # plt.errorbar(popt[3], 1, xerr=pcov[3, 3] ** 0.5, fmt='.', color=line_style(f)["color"])
    print(fix_temp)

ax.set_xlabel("Kinetic Energy [eV]")
ax.set_ylabel("Intensity [Au.]")
ax.set_title("Fermi Level Fit")
# ax.set_xlim(44.4, 44.55)
ax.legend()

# One checkbox per scan, hides/shows its data and fit together
check_ax = fig.add_axes([0.87, 0.5, 0.12, 0.05 * len(curve_lines)])
check_buttons = CheckButtons(check_ax, list(curve_lines), [True] * len(curve_lines))

def toggle_curve(lab):
    for line in curve_lines[lab]:
        line.set_visible(not line.get_visible())
    fig.canvas.draw_idle()

check_buttons.on_clicked(toggle_curve)
plt.show()

20.0
20.0
40.0
40.0
60.0


In [247]:
#This needs to use the not fixed temp
%matplotlib qt
from matplotlib.widgets import CheckButtons

from scipy.ndimage import gaussian_filter1d, zoom
from P04_Parser.util_funcs import (
    fit_2d_polynomial,
    evaluate_2d_polynomial,
    fermi_dirac,
    fermi_lin,
    angle_to_kx_deflector,
    angle_to_ky_deflector,
)
from scipy.optimize import curve_fit

colors = ["black", "navy", "cornflowerblue", "hotpink", "red", "darkorange", "gold"]
temp_to_index = {20: 0, 30: 1, 40: 2, 50: 3, 60: 4, 100: 5, 200: 6}


def line_style(file):
    name = Path(file).name
    T = temp_to_index[int(re.search(r"_(\d+(?:\.\d+)?)K", name).group(1))]
    return dict(
        color=colors[T],
        linestyle="--" if "down" in name.lower() else "-",
    )

fig, ax = plt.subplots(figsize=(14, 8))
fig.subplots_adjust(right=0.85)


fermi_levels = []
curve_lines = {}  # label -> [data line, fit line], toggled together by the check buttons

for f in files:
    fix_temp=sort_key(f)[0]

    data = load_ibw_swept_as_kmap(f)

    edc = data.get_EDC_inside(roi = hs.roi.CircleROI(cx=0, cy=0, r=15))
    span = hs.roi.SpanROI(left=-100, right=150)

    sig = span(edc)

    len_kin_en = sig.axes_manager[0].size
    offset_kin_en = sig.axes_manager[0].offset
    scale_kin_en = sig.axes_manager[0].scale

    kinetic_energies = np.linspace(
        offset_kin_en, offset_kin_en + scale_kin_en * len_kin_en, len_kin_en
    )

    intensity_data = sig.data
    
    intensity_data = (intensity_data - np.min(intensity_data)) / (np.max(intensity_data) - np.min(intensity_data))

    kinetic_energies, intensity_data = kinetic_energies[300:], intensity_data[300:]

    def gauss_fermi_lin(x, Gnd, Scale, Lin, Ef, kT, sigma):
        return gaussian_filter1d(fermi_lin(x, Gnd, Scale, Lin, Ef, kT), sigma=sigma)
    
    fit_guess = (
                intensity_data[-1],
                intensity_data[0],
                intensity_data[1] - intensity_data[0],
                kinetic_energies[len(kinetic_energies) // 2],
                0.1,
                10,
    )
    fit_func = gauss_fermi_lin
    bounds = (
        (0, 0, -np.inf, -np.inf, 0, 0),
        (np.inf, np.inf, np.inf, np.inf, np.inf, np.inf),
    )

    popt, pcov = curve_fit(
        fit_func, kinetic_energies, intensity_data, p0=fit_guess, bounds=bounds, absolute_sigma=False
    )



    # plt.figure()
    (data_line,) = ax.plot(kinetic_energies, intensity_data,
            label=labels[Path(f).name],
            **line_style(f),)
    kin_en_highres = np.linspace(
        kinetic_energies[0], kinetic_energies[-1], 1000
    )

    ## The sigma broadening in gaussian_filter1d works in pixel by pixel, so when we want to plot higher
    ## resolution, eg 1000 datapoints here, we need to scale up the sigma acordingly.
    (fit_line,) = ax.plot(
        kin_en_highres,
        fit_func(
            kin_en_highres, *popt[:-1], popt[-1] * 1000 / len(kinetic_energies)
        ),
        # label=labels[Path(f).name] + "fit",
        alpha=0.5,
        **line_style(f),
    )
    curve_lines[labels[Path(f).name]] = [data_line, fit_line]
    fermi_levels.append([popt[3], np.sqrt(pcov[3, 3]), labels[Path(f).name]])
    # plt.errorbar(popt[3], 1, xerr=pcov[3, 3] ** 0.5, fmt='.', color=line_style(f)["color"])
    print(fix_temp, round(popt[4]/8.6173303e-5, 2))

ax.set_xlabel("Kinetic Energy [eV]")
ax.set_ylabel("Intensity [Au.]")
ax.set_title("Fermi Level Fit")
# ax.set_xlim(44.4, 44.55)
ax.legend()

# One checkbox per scan, hides/shows its data and fit together
check_ax = fig.add_axes([0.87, 0.5, 0.12, 0.05 * len(curve_lines)])
check_buttons = CheckButtons(check_ax, list(curve_lines), [True] * len(curve_lines))

def toggle_curve(lab):
    for line in curve_lines[lab]:
        line.set_visible(not line.get_visible())
    fig.canvas.draw_idle()

check_buttons.on_clicked(toggle_curve)
plt.show()

20.0 38.22
20.0 37.87
40.0 55.11
40.0 54.77
60.0 68.61


In [248]:
# Scan start/end times straight from the files: start from the "Time=" entry in the ibw note,
# end from the file's modification time (written when the scan finishes)
from datetime import datetime
from scipy.interpolate import interp1d

def scan_window(file):
    """(start, end) of a scan in hours since midnight."""
    note = binarywave.load(file)["wave"]["note"]
    h, m, s = map(int, re.search(rb"Time=(\d\d):(\d\d):(\d\d)", note).groups())
    end = datetime.fromtimestamp(os.path.getmtime(file))
    return h + m / 60 + s / 3600, end.hour + end.minute / 60 + end.second / 3600

def temp_color(lab):
    return colors[temp_to_index[int(lab.split("K")[0])]]

label_to_file = {labels[Path(f).name]: f for f in files}
ta_window = {lab: scan_window(f) for lab, f in label_to_file.items()}

fesi_scan_files = [
    os.path.join(file_dir, f) for f in os.listdir(file_dir)
    if f.endswith(".ibw") and f.startswith("FeSi") and "swept" in f.lower()
]
fesi_window = {temp_from_filename(Path(f).name): scan_window(f) for f in fesi_scan_files}

# Ta foil points sit at the middle of their scan
rows = sorted((np.mean(ta_window[lab]), e, s, lab) for e, s, lab in fermi_levels)

# FeSi scans without a Ta foil measurement at the same temperature get E_F by linear interpolation
# of the Ta foil Fermi level in time, at the middle of the FeSi scan. Used by the Fermi level adjusted plot.
interp_labels = ["30K", "50K"]
ta_t = np.array([r[0] for r in rows])
E_F_vs_time = interp1d(ta_t, [r[1] for r in rows], kind="linear", fill_value="extrapolate")
E_F_err_vs_time = interp1d(ta_t, [r[2] for r in rows], kind="linear", fill_value="extrapolate")
fesi_E_F = {lab: float(E_F_vs_time(np.mean(fesi_window[lab]))) for lab in interp_labels}
fesi_E_F_err = {lab: float(E_F_err_vs_time(np.mean(fesi_window[lab]))) for lab in interp_labels}
for lab in interp_labels:
    print(f"FeSi {lab} interpolated E_F = {fesi_E_F[lab]:.4f} +- {fesi_E_F_err[lab]:.4f} eV")

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot([r[0] for r in rows], [r[1] for r in rows], "-", color="0.6", zorder=0)
for t, e, s, lab in rows:
    start, end = ta_window[lab]
    ax.errorbar(t, e, yerr=s, xerr=[[t - start], [end - t]], fmt="o", capsize=3,
                color=line_style(label_to_file[lab])["color"])
    ax.annotate(lab, (t, e), textcoords="offset points", xytext=(5, 6), fontsize=9)

# FeSi scans as shaded bands spanning the whole scan, labelled along the top of the axes
for lab, (start, end) in sorted(fesi_window.items(), key=lambda kv: kv[1]):
    ax.axvspan(start, end, color=temp_color(lab), alpha=0.15, lw=0, zorder=0)
    ax.text((start + end) / 2, 1.01, f"FeSi {lab}", transform=ax.get_xaxis_transform(),
            rotation=90, ha="center", va="bottom", fontsize=8, color=temp_color(lab))

# Interpolated Fermi levels for the FeSi scans without a matching Ta foil measurement
for lab in interp_labels:
    ax.errorbar(np.mean(fesi_window[lab]), fesi_E_F[lab], yerr=fesi_E_F_err[lab],
                fmt="D", mfc="none", capsize=3, ms=7, color=temp_color(lab))
ax.fill_between([], [], color="0.4", alpha=0.15, label="FeSi scan duration")
ax.plot([], [], "o", color="0.4", label="Ta foil Fermi level (bar = scan duration)")
ax.plot([], [], "D", mfc="none", color="0.4", label="Interpolated Fermi level")
ax.legend(loc="best")

ax.xaxis.set_major_formatter(
    mpl.ticker.FuncFormatter(lambda h, _: f"{int(h):02d}:{round(h % 1 * 60):02d}")
)
ax.ticklabel_format(axis="y", useOffset=False)
ax.set_xlabel("Time of measurement")
ax.set_ylabel("Fermi level (eV)")
fig.tight_layout()
plt.show()  


FeSi 30K interpolated E_F = 44.4586 +- 0.0000 eV
FeSi 50K interpolated E_F = 44.4578 +- 0.0000 eV


# Kinetic energy plot

In [ ]:
#Kinetic energy plot

%matplotlib qt
from matplotlib.widgets import Slider, Button, SpanSelector, CheckButtons

# Interactive FeSi EDCs on an E - E_F axis, with E_F from the Ta foil fit at the same temperature label
max_shift = 0.02  # eV, slider range is +-max_shift
norm_range = (44.1, 44.2)  # initial normalization band, E - E_F (eV)
E_F_fallback = 0#np.mean([ef for ef, _, _ in fermi_levels])  # eV, used when no Ta foil fit exists for that label
keep = None  # e.g. ["20K", "40K", "40K down"]; None shows all FeSi scans

fesi_names = sorted(
    [f for f in os.listdir(file_dir) if f.endswith(".ibw") and f.startswith("FeSi") and "swept" in f.lower()],
    key=sort_key,
)
if keep is not None:
    fesi_names = [f for f in fesi_names if temp_from_filename(f) in keep]
fesi_files = [os.path.join(file_dir, f) for f in fesi_names]

E_F_of = {lab: ef for ef, _, lab in fermi_levels}

edcs = []
slider_labels = []
for file in fesi_files:
    lab = temp_from_filename(Path(file).name)
    E_F = E_F_of.get(lab, E_F_fallback)

    data = load_ibw_swept_as_kmap(file)
    edc = data.get_EDC_inside(hs.roi.CircleROI(cx=np.pi/4.49, cy=0, r=2))
    edcs.append((file, edc.axes_manager[0].axis, edc.data))
    slider_labels.append(lab if lab in E_F_of else f"{lab}")

slider_height = 0.03
plot_bottom = 0.1 + slider_height * len(edcs)
fig, ax = plt.subplots(figsize=(14, 10))
fig.subplots_adjust(bottom=plot_bottom, right=0.85)

lines = []
for (file, energy, intensity), lab in zip(edcs, slider_labels):
    (line,) = ax.plot(energy, intensity, label=lab, **line_style(file))
    lines.append(line)

# ax.axvline(0, color="k", lw=0.8)
ax.set_xlim(np.min([np.min(energy) for _, energy, _ in edcs]), np.max([np.max(energy) for _, energy, _ in edcs]))
ax.set_xlabel(r"Kinetic Energy $E$ (eV)")
ax.set_ylabel("Normalized intensity")
ax.set_title("FeSi EDCs, Fermi level from Ta foil (drag the grey band to change the normalization range)")
ax.legend()

sliders = []
for i, lab in enumerate(slider_labels):
    slider_ax = fig.add_axes([0.2, plot_bottom - 0.06 - slider_height * (i + 1), 0.6, slider_height * 0.7])
    slider = Slider(
        slider_ax, lab, -max_shift * 1e3, max_shift * 1e3, valinit=0, valstep=0.1,
        valfmt="%+.1f meV", color=line_style(edcs[i][0])["color"],
    )
    sliders.append(slider)

def update(_=None):
    """Shift each curve by its slider, then normalize it to its mean inside the current band."""
    lo, hi = norm_selector.extents
    for (_, energy, intensity), line, slider in zip(edcs, lines, sliders):
        shifted = energy + slider.val * 1e-3
        in_range = (shifted >= lo) & (shifted <= hi)
        line.set_data(shifted, intensity / np.nanmean(intensity[in_range]) if in_range.any() else intensity * np.nan)
    ax.relim(visible_only=True)
    ax.autoscale_view(scalex=False)
    fig.canvas.draw_idle()

norm_selector = SpanSelector(
    ax,
    lambda lo, hi: update(),
    "horizontal",
    onmove_callback=lambda lo, hi: update(),
    interactive=True,
    drag_from_anywhere=True,
    props=dict(facecolor="0.6", alpha=0.3),
)
norm_selector.extents = norm_range

for slider in sliders:
    slider.on_changed(update)

# One checkbox per scan, hides/shows that curve
check_ax = fig.add_axes([0.87, plot_bottom + 0.1, 0.12, 0.04 * len(lines)])
check_buttons = CheckButtons(check_ax, slider_labels, [True] * len(lines))

def toggle_curve(lab):
    line = lines[slider_labels.index(lab)]
    line.set_visible(not line.get_visible())
    update()

check_buttons.on_clicked(toggle_curve)

reset_ax = fig.add_axes([0.9, 0.02, 0.08, 0.04])
reset_button = Button(reset_ax, "Reset")

def reset(_):
    for s in sliders:
        s.reset()
    norm_selector.extents = norm_range
    update()

reset_button.on_clicked(reset)

def get_shifts():
    """Current slider shift (meV) per curve, keyed by slider label."""
    return {lab: s.val for lab, s in zip(slider_labels, sliders)}

def get_norm_range():
    """Current normalization range (E - E_F, eV)."""
    return norm_selector.extents

update()
plt.show()
print(np.mean([ef for ef, _, _ in fermi_levels]))

44.4581097608703


# Fermi level adjusted plot

In [249]:
# Fermi level adjusted plot
%matplotlib qt
from matplotlib.widgets import Slider, Button, SpanSelector, CheckButtons

# Interactive FeSi EDCs on an E - E_F axis. E_F comes from the Ta foil fit at the same temperature label;
# labels without a Ta foil fit (30K, 50K) use the time-interpolated E_F (fesi_E_F, from the Fermi level over time cell)
max_shift = 0.02  # eV, slider range is +-max_shift
norm_range = (-0.2, -0.1)  # initial normalization band, E - E_F (eV)
E_F_fallback = np.mean([ef for ef, _, _ in fermi_levels])  # eV, used when neither a Ta foil fit nor an interpolated value exists
keep = None  # e.g. ["20K", "40K", "40K down"]; None shows all FeSi scans

fesi_names = sorted(
    [f for f in os.listdir(file_dir) if f.endswith(".ibw") and f.startswith("FeSi") and "swept" in f.lower()],
    key=sort_key,
)
if keep is not None:
    fesi_names = [f for f in fesi_names if temp_from_filename(f) in keep]
fesi_files = [os.path.join(file_dir, f) for f in fesi_names]

E_F_ta = {lab: ef for ef, _, lab in fermi_levels}
E_F_of = {**fesi_E_F, **E_F_ta}  # Ta foil fit wins where it exists

edcs = []
slider_labels = []
for file in fesi_files:
    lab = temp_from_filename(Path(file).name)
    E_F = E_F_of.get(lab, E_F_fallback)

    data = load_ibw_swept_as_kmap(file)
    edc = data.get_EDC_inside(hs.roi.CircleROI(cx=np.pi/4.49, cy=0, r=2))
    edcs.append((file, edc.axes_manager[0].axis - E_F, edc.data))
    if lab in E_F_ta:
        slider_labels.append(lab)
    elif lab in E_F_of:
        slider_labels.append(f"{lab} (interp. E_F)")
    else:
        slider_labels.append(f"{lab} (mean E_F)")

slider_height = 0.03
plot_bottom = 0.1 + slider_height * len(edcs)
fig, ax = plt.subplots(figsize=(14, 10))
fig.subplots_adjust(bottom=plot_bottom, right=0.85)

lines = []
for (file, energy, intensity), lab in zip(edcs, slider_labels):
    (line,) = ax.plot(energy, intensity, label=lab, **line_style(file))
    lines.append(line)

ax.axvline(0, color="k", lw=0.8)
ax.set_xlim(-0.25, 0.25)
ax.set_xlabel(r"$E - E_F$ (eV)")
ax.set_ylabel("Normalized intensity")
ax.set_title("FeSi EDCs, Fermi level from Ta foil (drag the grey band to change the normalization range)")
ax.legend()

sliders = []
for i, lab in enumerate(slider_labels):
    slider_ax = fig.add_axes([0.2, plot_bottom - 0.06 - slider_height * (i + 1), 0.6, slider_height * 0.7])
    slider = Slider(
        slider_ax, lab, -max_shift * 1e3, max_shift * 1e3, valinit=0, valstep=0.1,
        valfmt="%+.1f meV", color=line_style(edcs[i][0])["color"],
    )
    sliders.append(slider)

def update(_=None):
    """Shift each curve by its slider, then normalize it to its mean inside the current band."""
    lo, hi = norm_selector.extents
    for (_, energy, intensity), line, slider in zip(edcs, lines, sliders):
        shifted = energy + slider.val * 1e-3
        in_range = (shifted >= lo) & (shifted <= hi)
        line.set_data(shifted, intensity / np.nanmean(intensity[in_range]) if in_range.any() else intensity * np.nan)
    ax.relim(visible_only=True)
    ax.autoscale_view(scalex=False)
    fig.canvas.draw_idle()

norm_selector = SpanSelector(
    ax,
    lambda lo, hi: update(),
    "horizontal",
    onmove_callback=lambda lo, hi: update(),
    interactive=True,
    drag_from_anywhere=True,
    props=dict(facecolor="0.6", alpha=0.3),
)
norm_selector.extents = norm_range

for slider in sliders:
    slider.on_changed(update)

# One checkbox per scan, hides/shows that curve
check_ax = fig.add_axes([0.87, plot_bottom + 0.1, 0.12, 0.04 * len(lines)])
check_buttons = CheckButtons(check_ax, slider_labels, [True] * len(lines))

def toggle_curve(lab):
    line = lines[slider_labels.index(lab)]
    line.set_visible(not line.get_visible())
    update()

check_buttons.on_clicked(toggle_curve)

reset_ax = fig.add_axes([0.9, 0.02, 0.08, 0.04])
reset_button = Button(reset_ax, "Reset")

def reset(_):
    for s in sliders:
        s.reset()
    norm_selector.extents = norm_range
    update()

reset_button.on_clicked(reset)

def get_shifts():
    """Current slider shift (meV) per curve, keyed by slider label."""
    return {lab: s.val for lab, s in zip(slider_labels, sliders)}

def get_norm_range():
    """Current normalization range (E - E_F, eV)."""
    return norm_selector.extents

update()
plt.show()

In [200]:
# %matplotlib qt
from matplotlib.widgets import Slider, Button, SpanSelector

max_shift = 0.02  # eV, slider range is +-max_shift



# Uses keep, norm_range, E_F_of and E_F_fallback from the cell above
edcs = []
for file in files:
    lab = labels[Path(file).name]
    if lab not in keep:
        continue
    E_F = E_F_of.get(lab, E_F_fallback)

    data = load_ibw_swept_as_kmap(file)
    edc = data.get_EDC_inside(hs.roi.CircleROI(cx=0, cy=0, r=2))
    edcs.append((file, edc.axes_manager[0].axis - E_F, edc.data))

# Two files share the "20K" label, so number duplicates to keep slider names unique
slider_labels = []
for file, _, _ in edcs:
    lab = labels[Path(file).name]
    if lab not in E_F_of:
        lab += " (mean E_F)"
    n = sum(l == lab or l.startswith(lab + " #") for l in slider_labels)
    slider_labels.append(lab if n == 0 else f"{lab} #{n + 1}")

slider_height = 0.03
plot_bottom = 0.1 + slider_height * len(edcs)
fig, ax = plt.subplots(figsize=(14, 10))
fig.subplots_adjust(bottom=plot_bottom)

plt.rcParams.update({
    "font.size": 16,          # base size (also used for slider labels and values)
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})


lines = []
for (file, energy, intensity), lab in zip(edcs, slider_labels):
    (line,) = ax.plot(energy, intensity, label=lab, **line_style(file))
    lines.append(line)

ax.axvline(0, color="k", lw=0.8)
ax.set_xlim(-0.25, 0.25)
ax.set_xlabel(r"$E - E_F$ (eV)")
ax.set_ylabel("Normalized intensity")
ax.set_title("FeSi EDCs, Fermi level from Ta foil (drag the grey band to change the normalization range)")
ax.legend()

sliders = []
for i, lab in enumerate(slider_labels):
    slider_ax = fig.add_axes([0.2, plot_bottom - 0.06 - slider_height * (i + 1), 0.6, slider_height * 0.7])
    slider = Slider(
        slider_ax, lab, -max_shift * 1e3, max_shift * 1e3, valinit=0, valstep=0.1,
        valfmt="%+.1f meV", color=line_style(edcs[i][0])["color"],
    )

    sliders.append(slider)

def update(_=None):
    """Shift each curve by its slider, then normalize it to its mean inside the current band."""
    lo, hi = norm_selector.extents
    for (_, energy, intensity), line, slider in zip(edcs, lines, sliders):
        shifted = energy + slider.val * 1e-3
        in_range = (shifted >= lo) & (shifted <= hi)
        line.set_data(shifted, intensity / np.nanmean(intensity[in_range]) if in_range.any() else intensity * np.nan)
    ax.relim()
    ax.autoscale_view(scalex=False)
    fig.canvas.draw_idle()

norm_selector = SpanSelector(
    ax,
    lambda lo, hi: update(),
    "horizontal",
    onmove_callback=lambda lo, hi: update(),
    interactive=True,
    drag_from_anywhere=True,
    props=dict(facecolor="0.6", alpha=0.3),
)
norm_selector.extents = norm_range

for slider in sliders:
    slider.on_changed(update)

reset_ax = fig.add_axes([0.9, 0.02, 0.08, 0.04])
reset_button = Button(reset_ax, "Reset")

def reset(_):
    for s in sliders:
        s.reset()
    norm_selector.extents = norm_range
    update()

reset_button.on_clicked(reset)

def get_shifts():
    """Current slider shift (meV) per curve, keyed by slider label."""
    return {lab: s.val for lab, s in zip(slider_labels, sliders)}

def get_norm_range():
    """Current normalization range (E - E_F, eV)."""
    return norm_selector.extents

update()
plt.show()

TypeError: argument of type 'NoneType' is not iterable